In [50]:
import pathlib
import re

import numpy as np
import pandas as pd

from pyarrow import parquet as pq

from MDAnalysis.analysis.rms import rmsd

In [2]:
quantum_green = pathlib.Path("/home/shared/projects/quantum_green")
paquets = pathlib.Path("./parquets")

In [3]:
def xyz_str_to_array(xyz_str):
    rows = []
    for s in xyz_str.split("\n"):
        try:
            lst = s.split()
            rows.append([float(lst[i]) for i in [1, 2, 3]])
        except Exception:
            pass
    return np.array(rows)


def last_conformer_to_array(xyz):
    if xyz[-1].dtype != np.dtype("float32"):
        xyz = xyz[-1]
    return np.stack(xyz)[:, 3:]


SEMIEMPIRICAL_DFT_MAPPING = {
    ".tar": ".log",
    "/data1/": "/home/gridsan/",
    "semiempirical_opt": "DFT_opt_freq",
}


REGEX = re.compile(r"/(\d+)/\1.log")


def postprocess(source):
    for k, v in SEMIEMPIRICAL_DFT_MAPPING.items():
        source = source.replace(k, v)
    result = REGEX.search(source)
    if result:
        g1 = result.group(1)
        return source.replace(f"/{g1}/{g1}.", f"/{g1}.")
    else:
        return source

Read and process the consolidated species data

In [4]:
species_data = pd.read_pickle(
    quantum_green / "reactants" / "quantum_green_species_data_24march12b.pkl"
)
species_data["source"] = species_data["species_dft_log_path"].apply(postprocess)
species_data["xyz"] = species_data["xyz_str"].apply(xyz_str_to_array)

In [ ]:
assert species_data["species_id"].nunique() == len(species_data)
species_data["species_id"].min(), species_data["species_id"].max()

Read and process the parsed DFT data

In [6]:
dft = pq.read_table(
    paquets / "dft_nonts_dft_v9.parquet", columns=["source", "initial_xyz", "xyz"]
).to_pandas()
dft["source"] = dft["source"].apply(postprocess)
dft["initial_xyz"] = dft["initial_xyz"].apply(last_conformer_to_array)
dft["xyz"] = dft["xyz"].apply(last_conformer_to_array)
dft["dft_index"] = dft.index

In [7]:
filtered_dft = dft[dft["source"].isin(species_data["source"])].copy()
filtered_dft["species_xyz"] = filtered_dft["source"].map(
    species_data.set_index("source")["xyz"]
)
filtered_dft["rmsd"] = np.vectorize(rmsd)(
    filtered_dft["xyz"], filtered_dft["species_xyz"]
)

In [8]:
matched_dft = filtered_dft.loc[
    filtered_dft.groupby("source")["rmsd"].idxmin()
].reset_index(drop=True)
matched_dft["species_id"] = matched_dft["source"].map(
    species_data.set_index("source")["species_id"]
)

In [ ]:
matched_dft["rmsd"].max()

Assign the initial xyz to each species data point

In [10]:
species_data["initial_xyz"] = species_data["source"].map(
    matched_dft.set_index("source")["initial_xyz"]
)

Read and process the semiempirical data

In [11]:
semiempirical = pq.read_table(
    paquets / "semiempirical_nonts_semiempirical_v9.parquet",
    columns=["source", "initial_xyz", "std_xyz"],
).to_pandas().rename(columns={"std_xyz": "final_xyz"})
semiempirical["source"] = semiempirical["source"].apply(postprocess)
semiempirical["initial_xyz"] = semiempirical["initial_xyz"].apply(last_conformer_to_array)
semiempirical["final_xyz"] = semiempirical["final_xyz"].apply(last_conformer_to_array)
semiempirical["semiempirical_index"] = semiempirical.index

In [85]:
df = filtered_semiempirical = semiempirical[
    semiempirical["source"].isin(species_data["source"])
].copy()
df["dft_initial_xyz"] = df["source"].map(matched_dft.set_index("source")["initial_xyz"])


def msd_func(x, y):
    msd = np.square(x - y).sum(-1).mean()
    return msd if msd > 1e-8 else 0.0


msd = {
    case: np.vectorize(msd_func)(
        df[f"{case}_xyz"], df["dft_initial_xyz"]
    )
    for case in ["initial", "final"]
}

df["rmsd"] = np.sqrt(np.minimum(msd["initial"], msd["final"]))
df["from_initial"] = msd["initial"] < msd["final"]

In [86]:
df = matched_semiempirical = filtered_semiempirical.loc[
    filtered_semiempirical.groupby("source")["rmsd"].idxmin()
].reset_index(drop=True)
df["species_id"] = df["source"].map(species_data.set_index("source")["species_id"])

In [ ]:
matched_semiempirical["rmsd"].describe(percentiles=[0.999])

In [ ]:
matched_semiempirical["from_initial"].value_counts()

In [ ]:
df = matched_semiempirical[matched_semiempirical["from_initial"]]
total = 0
for case in ["aug11b", "sep1a_filtered", "co2_capture"]:
    num = sum(case in s for s in df["source"])
    total += num
    print(case, num)
print("total", total, len(df))

In [ ]:
df = matched_semiempirical[matched_semiempirical["rmsd"] > 0.0].copy()
total = 0
for case in ["aug11b", "sep1a_filtered", "co2_capture"]:
    num = sum(case in s for s in df["source"])
    total += num
    print(case, num)
print("total", total, len(df))

Save the results

In [93]:
df = matched_semiempirical
df["dft_index"] = df["source"].map(matched_dft.set_index("source")["dft_index"])
df[["species_id", "dft_index", "semiempirical_index", "rmsd", "from_initial"]].sort_values(
    "species_id"
).to_csv("quantum_green_species_data_match.csv", index=False)